使用PyTorch的整个流程可以拆解为 **5 个核心步骤**。


### 1. 数据准备 (Data Preparation)

PyTorch 使用两个核心类来处理数据：
*   **`Dataset`**: 负责封装数据（如图片、文本），并定义如何读取单个样本。
*   **`DataLoader`**: 负责将数据打包成批次（Batch），支持多线程加载和数据打乱（Shuffle）。

```python
import torch
from torch.utils.data import DataLoader, TensorDataset

# 1. 构造假数据 (实际中通常用 torchvision.datasets.MNIST 等)
# 1000个样本，每个样本10个特征
X_train = torch.randn(1000, 10)
y_train = torch.randint(0, 2, (1000,)) # 0或1的分类标签

# 2. 封装成 Dataset
train_dataset = TensorDataset(X_train, y_train)

# 3. 加载成 DataLoader
# batch_size=32 表示每次训练取32个样本
# shuffle=True 表示每个 epoch 都会打乱数据顺序
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
```


### 2. 定义网络结构 (Model Definition)

在 PyTorch 中，所有模型都必须继承自 `nn.Module` 类。你需要实现两个方法：
*   **`__init__`**: 定义网络层（如全连接层、卷积层）。
*   **`forward`**: 定义数据流向（前向传播逻辑）。

```python
import torch.nn as nn
import torch.nn.functional as F

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 定义层
        self.fc1 = nn.Linear(10, 64)  # 输入10 -> 隐藏层64
        self.fc2 = nn.Linear(64, 2)   # 隐藏层64 -> 输出2

    def forward(self, x):
        # 定义前向传播
        x = self.fc1(x)
        x = F.relu(x)  # 激活函数
        x = self.fc2(x)
        return x

# 实例化模型
model = SimpleNet()
```


### 3. 配置训练工具 (Setup)

在开始训练前，你需要准备好三样东西：**损失函数**、**优化器** 和 **设备（CPU/GPU）**。

```python
import torch.optim as optim

# 1. 选择设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # 将模型移动到 GPU

# 2. 定义损失函数 (例如二分类用 BCEWithLogitsLoss)
criterion = nn.BCEWithLogitsLoss()

# 3. 定义优化器 (例如 Adam)
optimizer = optim.Adam(model.parameters(), lr=0.001)
```

---

### 4. 训练循环 (Training Loop)

这是 PyTorch 最核心的部分。标准的训练流程包含 **5 个固定步骤**：

1.  **前向传播**：计算预测值。
2.  **计算损失**：对比预测值和真实值。
3.  **梯度清零**：清空上一轮的梯度（非常重要！）。
4.  **反向传播**：计算梯度。
5.  **参数更新**：优化器根据梯度更新权重。

```python
epochs = 10

for epoch in range(epochs):
    model.train()  # 切换到训练模式（启用 Dropout 等）
    
    for batch_idx, (data, target) in enumerate(train_loader):
        # 数据也要移动到 GPU
        data, target = data.to(device), target.to(device)
        
        # --- 核心五步法 ---
        
        # 1. 前向传播
        output = model(data)
        
        # 2. 计算损失
        loss = criterion(output, target)
        
        # 3. 梯度清零 (PyTorch 默认会累积梯度，所以每轮必须清零)
        optimizer.zero_grad()
        
        # 4. 反向传播 (自动计算梯度)
        loss.backward()
        
        # 5. 更新参数
        optimizer.step()
        
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")
```

### 5. 测试与评估 (Testing & Evaluation)

测试时，我们不需要计算梯度，因此需要使用 `torch.no_grad()` 来节省内存，并将模型切换到评估模式 `model.eval()`。

```python
def test():
    model.eval()  # 切换到评估模式（关闭 Dropout）
    correct = 0
    total = 0
    
    with torch.no_grad():  # 禁用梯度计算
        for data, target in train_loader: # 这里用训练集演示，实际应用测试集
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            # 获取预测结果 (假设输出是概率或 logits)
            _, predicted = torch.max(output.data, 1) # 使用torch内置函数来寻找tensor张量中的最大值，返回最大值和其所在的索引。
            
            total += target.size(0)
            correct += (predicted == target).sum().item()

    print(f"Accuracy: {100 * correct / total}%")

test()
```

需要强调的是，训练、测试时获取的数据都是**tensor张量**，所以我们尽量使用torch内置函数来处理数据，比如这里的`torch.max()`。

### 常见处理方式速查表

在实际项目中，你还会经常用到以下功能：

| 功能 | 代码/方法 | 说明 |
| :--- | :--- | :--- |
| **保存模型** | `torch.save(model.state_dict(), 'model.pth')` | **推荐**。只保存参数，加载时需先实例化模型类。 |
| **加载模型** | `model.load_state_dict(torch.load('model.pth'))` | 加载参数到已实例化的模型中。 |
| **保存整个模型** | `torch.save(model, 'model.pth')` | 不推荐，依赖具体的文件路径和类定义。 |
| **早停法** | 手动监控验证集 Loss | 当验证集 Loss 不再下降时，停止训练。 |
| **学习率调整** | `torch.optim.lr_scheduler.StepLR` | 每隔几个 epoch 自动降低学习率。 |
| **迁移学习** | `torchvision.models.resnet50(pretrained=True)` | 加载预训练模型，微调最后一层。 |

### 总结

PyTorch 的核心逻辑非常清晰：
1.  **数据** -> `DataLoader`
2.  **模型** -> `nn.Module`
3.  **训练** -> 前向 -> 损失 -> 反向 -> 更新
4.  **测试** -> `no_grad()` -> 评估

掌握这个流程，你就掌握了 PyTorch 90% 的用法！
